In [1]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/silva/.netrc.
wandb: Currently logged in as: silvapi1994 (silvapi1994-karabuk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Import the Weights & Biases library for experiment tracking and artifact versioning
import wandb

# Import pandas for DataFrame handling and CSV export
import pandas as pd 

# Import the California Housing dataset loader from sklearn
from sklearn.datasets import fetch_california_housing 


def load_data():
    # Initialize a W&B run
    # project: logical project name in W&B UI
    # job_type: label describing what this run does (here: dataset upload)
    # Using "with" ensures the run is properly closed after execution
    with wandb.init(project="house_price_pred", job_type="upload_dataset") as run:

        # -------------------------------
        # 1. FETCH DATA
        # -------------------------------
        # Download the California Housing dataset as a pandas DataFrame
        data = fetch_california_housing(as_frame=True)

        # Extract the DataFrame (features + target)
        df = data.frame 

        # -------------------------------
        # 2. SAVE DATA LOCALLY
        # -------------------------------
        # Define the local filename for the raw dataset
        file_name = "cali_house_raw.csv"

        # Save DataFrame to CSV without the index column
        df.to_csv(file_name, index=False) 

        # -------------------------------
        # 3. CREATE A W&B ARTIFACT
        # -------------------------------
        # Artifacts are versioned data objects in W&B (datasets, models, etc.)
        raw_data_artifact = wandb.Artifact(
            name="cali_house_raw",          # Logical artifact name
            type="dataset",                 # Artifact type (dataset/model/etc.)
            description="Raw Cali Housing Dataset",  # Human-readable description
            metadata={
                "source": "sklearn dataset" # Extra info for reproducibility
            }
        ) 

        # -------------------------------
        # 4. ADD FILE TO ARTIFACT
        # -------------------------------
        # Attach the local CSV file to the artifact
        raw_data_artifact.add_file(file_name)

        # -------------------------------
        # 5. LOG (UPLOAD) ARTIFACT TO W&B
        # -------------------------------
        # This uploads the artifact to W&B servers
        # and creates the first version (v0) of this dataset
        run.log_artifact(raw_data_artifact)

        # Console confirmation message
        print("Done uploading the artifact to W&B")


# Execute the function to run the full pipeline
load_data()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/silva/.netrc.
wandb: Currently logged in as: silvapi1994 (silvapi1994-karabuk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Done uploading the artifcat to wandb


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import pandas as pd
import wandb


def train():
    # --------------------------------------------------
    # Experiment configuration (tracked automatically by W&B)
    # --------------------------------------------------
    config = {
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 1,
        "random_state": 42,
        "test_size": 0.2
    }

    # --------------------------------------------------
    # Initialize W&B run
    # - project: logical experiment group
    # - job_type: pipeline stage (training)
    # - config: hyperparameters (versioned)
    # - tags: searchable metadata
    # --------------------------------------------------
    with wandb.init(
        project="house_price_pred",
        job_type="training",
        config=config,
        tags=["random_forest", "regression", "housing"]
    ) as run:

        # --------------------------------------------------
        # 1. LOAD DATA FROM ARTIFACT (versioned dataset)
        # Ensures reproducibility (data hash + lineage)
        # --------------------------------------------------
        raw_data_artifact = run.use_artifact("cali_house_raw:latest")
        artifact_dir = raw_data_artifact.download()
        data_path = f"{artifact_dir}/cali_house_raw.csv"

        df = pd.read_csv(data_path)

        # Select numeric features only (RF requires numeric)
        x = df.drop("MedHouseVal", axis=1).select_dtypes(include="number")
        y = df["MedHouseVal"]

        # --------------------------------------------------
        # 2. TRAIN / TEST SPLIT (deterministic via random_state)
        # Logged implicitly through config
        # --------------------------------------------------
        x_train, x_test, y_train, y_test = train_test_split(
            x,
            y,
            test_size=config["test_size"],
            random_state=config["random_state"]
        )

        # --------------------------------------------------
        # 3. MODEL INITIALIZATION
        # Only pass relevant hyperparameters to the estimator
        # (avoids leaking unrelated config keys)
        # --------------------------------------------------
        model = RandomForestRegressor(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            min_samples_leaf=config["min_samples_leaf"],
            random_state=config["random_state"]
        )

        # Fit model
        model.fit(x_train, y_train)

        # --------------------------------------------------
        # 4. EVALUATION (hold-out test set)
        # Metrics logged to W&B for comparison across runs
        # --------------------------------------------------
        y_pred = model.predict(x_test)

        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        wandb.log({
            "mse": mse,
            "r2": r2
        })

        # --------------------------------------------------
        # Diagnostic plots (residuals, predictions vs truth)
        # NOTE: argument order must be X_train, X_test, y_train, y_test
        # --------------------------------------------------
        wandb.sklearn.plot_regressor(
            model,
            x_train,
            x_test,
            y_train,
            y_test
        )

        # --------------------------------------------------
        # 5. SAVE MODEL LOCALLY
        # Joblib preserves sklearn pipeline state
        # --------------------------------------------------
        model_file = "random_forest_model.joblib"
        joblib.dump(model, model_file)

        # --------------------------------------------------
        # 6. LOG MODEL AS ARTIFACT (versioned + reproducible)
        # Metadata stores hyperparameters for lineage
        # --------------------------------------------------
        model_artifact = wandb.Artifact(
            name="random_forest_model",
            type="model",
            description="Random Forest Model for House Price Prediction",
            metadata=dict(config)
        )

        model_artifact.add_file(model_file)

        # Upload model artifact and link it to this run
        run.log_artifact(model_artifact)

        print("Done training the model")


train()


wandb:   1 of 1 files downloaded.  
wandb: 
wandb: Plotting Regressor.
/home/silva/SILVA.AI/Projects/MLOps/DDODS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
wandb: Logged summary metrics.
wandb: Logged learning curve.
/home/silva/SILVA.AI/Projects/MLOps/DDODS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
wandb: WARNING using only the first 1000 datapoints to create chart outlier_candidates
wandb: Logged outlier candidates.
/home/silva/SILVA.AI/Projects/MLOps/DDODS/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
wandb: WARNING using only the first 100 datap

Done training the model


mse,▁
r2,▁
mse,0.29649
r2,0.77374
